# Submission v9 — EDA Phasic/Tonic Decomposition + v7c Base

## Research basis
Literature on Empatica E4 stress detection consistently shows:
- Raw EDA mean/std are **weak** cross-person features (dominated by tonic baseline which is person-specific)
- **SCR peak count and phasic EDA std** are the #1 most important EDA features
- The phasic component (fast SCR bursts) is **person-invariant** — everyone's sympathetic nervous system fires the same way under stress regardless of their absolute EDA level
- Tonic EDA encodes who the person is; phasic EDA encodes stress state

## What's new in v9
Added 5 phasic EDA features via `neurokit2` highpass decomposition:
- `eda_phasic_std` — variability of fast EDA responses (key stress marker)
- `eda_phasic_max` — peak phasic response in window
- `eda_phasic_mean_abs` — average phasic activity level
- `eda_tonic_slope` — is arousal building (+) or declining (-) over window?
- `eda_n_scr_peaks` — count of Skin Conductance Response events (most cited feature)

Everything else identical to v7c (LOPO=0.5130, LB=0.37629, α=1.8).

## Gate
Check LOPO after extraction — only proceed with ensemble if LOPO ≥ 0.5130.


In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy neurokit2 -q


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
import neurokit2 as nk
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000; HALF_MS = 90_000; THIRD_MS = 60_000
FS = 32  # Empatica E4 sampling rate

def hrv_time_domain(bpm_series):
    """Time-domain HRV — robust and person-invariant."""
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']: f['hrv_'+k]=np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm)>=32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn']/f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def eda_phasic_features(eda_series):
    """
    Decompose EDA into tonic (slow baseline) + phasic (fast SCR bursts).
    Phasic component removes person-specific tonic baseline automatically.
    Key finding from Empatica E4 research: SCR peak count is the #1 stress marker.

    Method: highpass filter at 0.05Hz (fast, no external solver needed).
    """
    f = {}
    v = eda_series.dropna().values.astype(float)
    if len(v) < 64:
        for k in ['phasic_std','phasic_max','phasic_mean_abs','tonic_slope','n_scr_peaks']:
            f['eda_'+k] = np.nan
        return f
    try:
        eda_clean = nk.eda_clean(v, sampling_rate=FS)
        decomp    = nk.eda_phasic(eda_clean, sampling_rate=FS, method='highpass')
        phasic    = decomp['EDA_Phasic'].values
        tonic     = decomp['EDA_Tonic'].values

        f['eda_phasic_std']      = float(np.std(phasic))
        f['eda_phasic_max']      = float(np.max(phasic))
        f['eda_phasic_mean_abs'] = float(np.mean(np.abs(phasic)))

        # Tonic slope: is baseline arousal building or declining over window?
        t_norm = np.linspace(0, 1, len(tonic))
        f['eda_tonic_slope'] = float(np.polyfit(t_norm, tonic, 1)[0])

        # SCR peak count — #1 cited feature in E4 stress literature
        peaks_df, _ = nk.eda_peaks(phasic, sampling_rate=FS,
                                    method='neurokit', amplitude_min=0.005)
        f['eda_n_scr_peaks'] = int(peaks_df['SCR_Peaks'].sum()) if 'SCR_Peaks' in peaks_df.columns else 0

    except Exception:
        for k in ['phasic_std','phasic_max','phasic_mean_abs','tonic_slope','n_scr_peaks']:
            f['eda_'+k] = np.nan
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    n = len(label_df)
    for i, (_, lrow) in enumerate(label_df.iterrows()):
        if i % 100 == 0:
            print(f'  {i}/{n} windows processed...')
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]

        # Standard absolute stats (v7c foundation)
        for col in SENSOR_COLS:
            v  = wa[col].dropna().values.astype(float)
            vf = wf[col].dropna().values.astype(float)
            vl = wl[col].dropna().values.astype(float)
            vt1 = wt1[col].dropna().values.astype(float)
            vt3 = wt3[col].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{col}_{s}'] = np.nan
                continue
            feat[f'{col}_mean']   = float(np.mean(v));   feat[f'{col}_std']    = float(np.std(v))
            feat[f'{col}_min']    = float(np.min(v));    feat[f'{col}_max']    = float(np.max(v))
            feat[f'{col}_median'] = float(np.median(v))
            feat[f'{col}_skew']   = float(spstats.skew(v))     if len(v)>2 else 0.
            feat[f'{col}_kurt']   = float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[f'{col}_range']  = float(np.max(v)-np.min(v))
            feat[f'{col}_q25']    = float(np.percentile(v,25))
            feat[f'{col}_q75']    = float(np.percentile(v,75))
            feat[f'{col}_iqr']    = float(np.percentile(v,75)-np.percentile(v,25))
            feat[f'{col}_delta']  = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[f'{col}_slope']  = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[f'{col}_t1_mean']= float(np.mean(vt1)) if len(vt1)>0 else float(np.mean(v))
            feat[f'{col}_t3_mean']= float(np.mean(vt3)) if len(vt3)>0 else float(np.mean(v))
            feat[f'{col}_t3t1']   = feat[f'{col}_t3_mean'] - feat[f'{col}_t1_mean']

        # Accel magnitude
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag=np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean']=float(np.mean(mag))
            feat['accel_mag_std'] =float(np.std(mag))
            feat['accel_mag_max'] =float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan

        # Time-domain HRV
        feat.update(hrv_time_domain(wa['heart_rate']))

        # NEW: EDA phasic/tonic decomposition — the key addition
        feat.update(eda_phasic_features(wa['eda']))

        # Person ID encoding
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p:i for i,p in enumerate(TRAIN_LABEL['pid'].unique())}

print('Extracting train features (includes phasic EDA decomposition ~5min)...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print(f'  train shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print(f'  test shape: {test_features.shape}')

print(f'\nNew features added vs v7c (106): {train_features.shape[1] - 106}')
print('New features: eda_phasic_std, eda_phasic_max, eda_phasic_mean_abs, eda_tonic_slope, eda_n_scr_peaks')


Extracting train features (includes phasic EDA decomposition ~5min)...
  0/815 windows processed...
  100/815 windows processed...
  200/815 windows processed...
  300/815 windows processed...
  400/815 windows processed...
  500/815 windows processed...
  600/815 windows processed...
  700/815 windows processed...
  800/815 windows processed...
  train shape: (815, 111)
Extracting test features...
  0/1028 windows processed...
  100/1028 windows processed...
  200/1028 windows processed...
  300/1028 windows processed...
  400/1028 windows processed...
  500/1028 windows processed...
  600/1028 windows processed...
  700/1028 windows processed...
  800/1028 windows processed...
  900/1028 windows processed...
  1000/1028 windows processed...
  test shape: (1028, 111)

New features added vs v7c (106): 5
New features: eda_phasic_std, eda_phasic_max, eda_phasic_mean_abs, eda_tonic_slope, eda_n_scr_peaks


In [4]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {0: total/(n_cls*counts[0]),
                 1: min(total/(n_cls*counts[1]), 2.5),
                 2: total/(n_cls*counts[2])}
sample_weights = np.array([class_weights[yi] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class weights (capped):', {k:round(v,3) for k,v in class_weights.items()})

# Sanity check: verify phasic features have real values
phasic_cols = [c for c in X_imp.columns if 'phasic' in c or 'n_scr' in c or 'tonic_slope' in c]
print(f'\nPhasic EDA features ({len(phasic_cols)}):')
print(X_imp[phasic_cols].describe().round(4).to_string())


X_imp shape     : (815, 111)
X_test_imp shape: (1028, 111)
Class weights (capped): {0: 1.677, 1: 2.5, 2: 0.463}

Phasic EDA features (5):
       eda_phasic_std  eda_phasic_max  eda_phasic_mean_abs  eda_tonic_slope  eda_n_scr_peaks
count        815.0000        815.0000             815.0000         815.0000         815.0000
mean           0.1069          0.4246               0.0719           0.0330         167.7926
std            0.2179          0.7258               0.1608           2.4037          52.5158
min            0.0012          0.0041               0.0010         -16.0538          28.0000
25%            0.0045          0.0280               0.0030          -0.0576         128.0000
50%            0.0149          0.0857               0.0081          -0.0007         171.0000
75%            0.1158          0.5372               0.0732           0.0566         209.0000
max            1.8989          5.4153               1.4869          32.4689         294.0000


In [5]:
# Identical hyperparams to v7c — proven at LOPO=0.5130
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.02,
    num_leaves        = 127,
    max_depth         = -1,
    min_child_samples = 5,
    subsample         = 0.6,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.3,
    reg_lambda        = 0.3,
    class_weight      = 'balanced',
    objective         = 'multiclass',
    num_class         = 3,
    n_jobs            = -1,
    verbose           = -1,
)

print('=== LOPO CV — v9 (v7c + phasic EDA features) ===')
logo = LeaveOneGroupOut()
lopo_scores = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]
    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only 1 class'); continue

    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
    m.fit(X_imp.iloc[tr_idx], y.iloc[tr_idx],
          sample_weight = sample_weights[tr_idx],
          eval_set      = [(X_imp.iloc[val_idx], y_val)],
          callbacks     = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

    sc = balanced_accuracy_score(y_val, m.predict(X_imp.iloc[val_idx]))
    print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(val_idx)})')
    lopo_scores.append(sc)

lopo_mean = np.mean(lopo_scores)
print(f'\nv9 LOPO = {lopo_mean:.4f} +/- {np.std(lopo_scores):.4f}')
print('v7c ref  = 0.5130  (current best)')

if lopo_mean < 0.50:
    print('\n⚠️  LOPO dropped — phasic features hurt. Do NOT run ensemble.')
elif lopo_mean < 0.5130:
    print('\n⚠️  LOPO slightly lower than v7c. Borderline — check feature stats above.')
else:
    print('\n✅ LOPO improved — proceed with ensemble.')


=== LOPO CV — v9 (v7c + phasic EDA features) ===
  Leave out 43JW: 0.5000  (n=93)
  Leave out C8Q6: 0.4859  (n=152)
  Leave out DT5C: 0.5998  (n=90)
  Leave out F1ZM: 0.4963  (n=137)
  Leave out HDS9: 0.6368  (n=135)
  Leave out P4DZ: 0.2496  (n=144)
  Leave out TPQI: 0.5676  (n=64)

v9 LOPO = 0.5051 +/- 0.1171
v7c ref  = 0.5130  (current best)

⚠️  LOPO slightly lower than v7c. Borderline — check feature stats above.


In [6]:
# Only run if LOPO >= 0.5130 above
print('=== Final ensemble: 3 seeds x 5 folds ===')
SEEDS = [42, 7, 123]
all_test_proba = []; all_cv_scores = []

for seed in SEEDS:
    skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr = X_imp.iloc[tr_idx]; y_tr = y.iloc[tr_idx]
        X_va = X_imp.iloc[val_idx]; y_va = y.iloc[val_idx]

        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(X_tr, y_tr,
              sample_weight = sample_weights[tr_idx],
              eval_set      = [(X_va, y_va)],
              callbacks     = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])

        sc = balanced_accuracy_score(y_va, m.predict(X_va))
        fold_scores.append(sc)
        seed_proba += m.predict_proba(X_test_imp)
        print(f'  Seed {seed} Fold {fold+1}: val BA = {sc:.4f}')

    seed_proba /= 5
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV = {np.mean(fold_scores):.4f}')

print(f'\nEnsemble CV = {np.mean(all_cv_scores):.4f}')
print('v7c ref = 0.8115')

raw_proba   = np.mean(all_test_proba, axis=0)

# Fixed α=1.8 — validated on LB
CALIB_ALPHA = 1.8
cal_proba   = raw_proba * (train_prior ** CALIB_ALPHA)
cal_proba   = cal_proba / cal_proba.sum(axis=1, keepdims=True)
final_preds = np.argmax(cal_proba, axis=1).astype(int)

print(f'\nCalibration alpha = {CALIB_ALPHA} (fixed)')
print('Raw distribution:')
for u, cnt in zip(*np.unique(np.argmax(raw_proba,1), return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')
print('Calibrated:')
for u, cnt in zip(*np.unique(final_preds, return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')
print('Train prior:')
for cls in range(3): print(f'  class {cls}: {counts[cls]}  ({counts[cls]/total*100:.1f}%)')


=== Final ensemble: 3 seeds x 5 folds ===
  Seed 42 Fold 1: val BA = 0.8414
  Seed 42 Fold 2: val BA = 0.7664
  Seed 42 Fold 3: val BA = 0.8726
  Seed 42 Fold 4: val BA = 0.7461
  Seed 42 Fold 5: val BA = 0.8600
  Seed 42 mean CV = 0.8173
  Seed 7 Fold 1: val BA = 0.8179
  Seed 7 Fold 2: val BA = 0.8223
  Seed 7 Fold 3: val BA = 0.8254
  Seed 7 Fold 4: val BA = 0.8597
  Seed 7 Fold 5: val BA = 0.7821
  Seed 7 mean CV = 0.8215
  Seed 123 Fold 1: val BA = 0.8463
  Seed 123 Fold 2: val BA = 0.6682
  Seed 123 Fold 3: val BA = 0.8752
  Seed 123 Fold 4: val BA = 0.8352
  Seed 123 Fold 5: val BA = 0.7591
  Seed 123 mean CV = 0.7968

Ensemble CV = 0.8119
v7c ref = 0.8115

Calibration alpha = 1.8 (fixed)
Raw distribution:
  class 0: 390  (37.9%)
  class 1: 501  (48.7%)
  class 2: 137  (13.3%)
Calibrated:
  class 0: 161  (15.7%)
  class 1: 26  (2.5%)
  class 2: 841  (81.8%)
Train prior:
  class 0: 162  (19.9%)
  class 1: 66  (8.1%)
  class 2: 587  (72.0%)


In [7]:
submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission_v9.csv', index=False)
print('submission_v9.csv saved!')
print(submission.head(10))


submission_v9.csv saved!
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       2
5  1232       2
6  1233       0
7  1234       2
8  1235       0
9  1236       0
